[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1d0-paRZanuewG6yL09DlaJTx0vYxR0H3)

# In-Silico Target Identification in Bovine Mastitis Pathogens
### Interactive Workshop & Demonstration Pipeline

This interactive notebook demonstrates a complete subtractive genomics & reverse vaccinology workflow for discovering therapeutic targets against 4 mastitis-causing pathogens:
1. *Staphylococcus aureus* (Gram-positive)
2. *Streptococcus uberis* (Gram-positive)
3. *Escherichia coli* (Gram-negative)
4. *Klebsiella pneumoniae* (Gram-negative)

---
### Workflow Pipeline Architecture:
```
NCBI Proteomes (4 species) ──► OrthoFinder ──► Core Orthogroups (present in all 4)
                                     │
                                     ▼
                                   CD-HIT ──► Non-Redundant Core (100% identity)
                                     │
                                     ▼
                            Host BLASTP (Bovine/Human) ──► Host Non-Homologs
                                     │
                                     ▼
                                 DEG BLASTP ──► Essential Bacterial Genes
                                     │
                                     ▼
                                 TTD BLASTP ──► Druggable Therapeutic Targets
                                     │
                                     ▼
                            ProtParam (BioPython) ──► Stable Proteins (Instability < 40)
                                     │
                                     ▼
                             Gram Classification ──► Gram(+) & Gram(-) Candidate Sets
                                     │
                                     ▼
                              PSORTb Server ──► Subcellular Localization
```

> **Note on Workshop / Demo Mode:**
> Heavy bioinformatics steps (e.g. OrthoFinder, massive host BLAST searches) normally require substantial CPU hours. This notebook includes a **`FAST_DEMO_MODE`** switch. When enabled, it checks for precomputed results in the repository folders, explains the underlying command, and runs all downstream filtering, parsing, data transformation, and plotting cells **in real-time**.

## Step 0: Pull Data and Scripts from GitHub
The first step clones your workshop GitHub repository containing the input proteomes, precomputed results, and helper scripts.

In [ ]:
# =========================================================================
# 1. CLONE WORKSHOP REPOSITORY FROM GITHUB
# =========================================================================
import os

# Set your GitHub repository URL:
GITHUB_REPO_URL = "https://github.com/bioinfokushwaha/FIAB2026_DT.git"
REPO_NAME = "FIAB2026_DT"

if not os.path.exists(REPO_NAME):
    print(f"Pulling workshop data and precomputed results from {GITHUB_REPO_URL}...")
    !git clone {GITHUB_REPO_URL}
    if os.path.exists(REPO_NAME):
        %cd {REPO_NAME}
else:
    if os.path.basename(os.getcwd()) != REPO_NAME:
        %cd {REPO_NAME}
    print("Already in workshop directory:", os.getcwd())

# Set Execution Mode:
# Set to True for instantaneous workshop demonstration using precomputed intermediate files
# Set to False if you want to install and compute everything locally from scratch
FAST_DEMO_MODE = True
print(f"\nWorkshop mode initialized: FAST_DEMO_MODE = {FAST_DEMO_MODE}")

# Fallback safeguard: If precomputed demo results are missing, auto-generate them in 2 seconds
if not os.path.exists('04_Druggability/Core_HS_DEG_TTD_Stable.fasta'):
    print('Generating workshop demo files using local proteome sequences...')
    !python scripts/generate_demo_data.py


## Step 1: Install Python Packages & Tools
Installs lightweight libraries (`biopython`, `pandas`, `matplotlib`) needed for data processing and visualization.

In [ ]:
!pip install -q biopython pandas matplotlib

import pandas as pd
import matplotlib.pyplot as plt
import Bio
from Bio import SeqIO

print(f"BioPython version: {Bio.__version__}")
print(f"Pandas version:    {pd.__version__}")
print("Environment ready!")


## Step 2: Check Input Bacterial Proteomes
Verify that the 4 bacterial proteome FASTA files (`.faa`) are present in `Proteome/`.

In [ ]:
import glob
proteome_files = sorted(glob.glob("Proteome/*.faa"))

print(f"Total proteome files found: {len(proteome_files)}\n")
for f in proteome_files:
    count = sum(1 for line in open(f) if line.startswith('>'))
    print(f"- {os.path.basename(f):35s}: {count:,} protein sequences")


## Step 3: OrthoFinder - Homology & Orthogroup Identification
OrthoFinder groups all bacterial proteins into orthologous gene families (orthogroups).

**Production command:**
```bash
orthofinder -f Proteome -t 15 -a 15
```

In [ ]:
import glob, os, time

orthogroups_candidates = sorted(glob.glob("Proteome/OrthoFinder/Results_*/Orthogroups/Orthogroups.tsv"))

if FAST_DEMO_MODE and orthogroups_candidates:
    orthogroups_tsv = orthogroups_candidates[-1]
    print(f"[DEMO MODE] Loading precomputed OrthoFinder results from: {orthogroups_tsv}")
else:
    print("Running OrthoFinder (this may take 15-30 mins depending on CPU)... ")
    # If standalone orthofinder is present, run it
    !orthofinder -f Proteome -t 4 -a 4
    orthogroups_tsv = sorted(glob.glob("Proteome/OrthoFinder/Results_*/Orthogroups/Orthogroups.tsv"))[-1]

# Inspect first 3 rows of Orthogroups table
!head -n 3 {orthogroups_tsv}


## Step 4: Pangenome Classification (Core, Accessory, Unique)
We convert the orthogroup table into a presence/absence matrix and classify:
- **Core Orthogroups:** Present across all 4 species (`presence == 4`)
- **Accessory Orthogroups:** Present in 2 or 3 species
- **Unique Orthogroups:** Species-specific (present in only 1 species)

In [ ]:
df = pd.read_csv(orthogroups_tsv, sep="\t")
species_cols = df.columns[1:]

print("===== INPUT FILE CHECK =====")
print("Total orthogroups identified:", len(df))
print("Number of species:", len(species_cols))
print("Species analysed:", list(species_cols))

# Build binary matrix
binary_df = df.copy()
binary_df[species_cols] = df[species_cols].notna().astype(int)
binary_df.to_csv("Ortho_pangenome_matrix.csv", index=False)

# Count presence
presence_count = binary_df[species_cols].sum(axis=1)
num_species = len(species_cols)

core = binary_df[presence_count == num_species]
accessory = binary_df[(presence_count >= 2) & (presence_count < num_species)]
unique = binary_df[presence_count == 1]
unique_counts = unique[species_cols].sum()

# Save CSV tables
core.to_csv("Ortho_core_genes.csv", index=False)
accessory.to_csv("Ortho_accessory_genes.csv", index=False)
unique.to_csv("Ortho_unique_genes.csv", index=False)

print("\n===== PANGENOME SUMMARY =====")
print(f"Core orthogroups:      {len(core):,}")
print(f"Accessory orthogroups: {len(accessory):,}")
print(f"Unique orthogroups:    {len(unique):,}")

print("\n===== UNIQUE ORTHOGROUPS PER SPECIES =====")
print(unique_counts.to_string())


In [ ]:
# Visualize Pangenome Distribution
counts = {"Core": len(core), "Accessory": len(accessory), "Unique": len(unique)}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Composition Bar
axes[0].bar(counts.keys(), counts.values(), color=["#27ae60", "#2980b9", "#c0392b"], edgecolor="black")
axes[0].set_ylabel("Number of Orthogroups", fontsize=12)
axes[0].set_title("Pangenome Composition", fontsize=14, fontweight="bold")
for k, v in counts.items():
    axes[0].text(k, v + max(counts.values())*0.02, f"{v:,}", ha="center", fontweight="bold")

# 2. Unique Orthogroups per Species
unique_counts.plot(kind="bar", ax=axes[1], color="#8e44ad", edgecolor="black")
axes[1].set_ylabel("Number of Unique Orthogroups", fontsize=12)
axes[1].set_title("Unique Orthogroups per Species", fontsize=14, fontweight="bold")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha="right")
for i, v in enumerate(unique_counts):
    axes[1].text(i, v + max(unique_counts)*0.02, f"{v:,}", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("pangenome_composition_summary.png", dpi=300)
plt.show()


## Step 5: Extract Core Sequences & Remove Redundancy (CD-HIT 100%)
Extract protein sequences corresponding to core orthogroups and collapse 100% identical duplicates.

**Production command:**
```bash
cd-hit -i 02_Annotation/Core_OrthoGp_proteome.fa -o 02_Annotation/Core_OrthoGp_proteome_NR1.fa -c 1.0 -n 5
```

In [ ]:
import os
os.makedirs("02_Annotation/Core_OrthoGP_Seq", exist_ok=True)

# Extract core IDs
core_ids = core["Orthogroup"].tolist()
with open("02_Annotation/Core_Ortho_Ids", "w") as f:
    f.write("\n".join(core_ids))

print(f"Extracted {len(core_ids)} Core Orthogroup IDs.")

# Check if merged core FASTA exists or compile it
core_fa = "02_Annotation/Core_OrthoGp_proteome.fa"
ortho_seq_dirs = sorted(glob.glob("Proteome/OrthoFinder/Results_*/Orthogroup_Sequences"))

if not os.path.exists(core_fa) and ortho_seq_dirs:
    ortho_seq_dir = ortho_seq_dirs[-1]
    print("Compiling Core_OrthoGp_proteome.fa from Orthogroup_Sequences/...")
    with open(core_fa, "w") as out_f:
        for og in core_ids:
            og_file = os.path.join(ortho_seq_dir, f"{og}.fa")
            if os.path.exists(og_file):
                out_f.write(open(og_file).read())

nr1_fa = "02_Annotation/Core_OrthoGp_proteome_NR1.fa"
if os.path.exists(nr1_fa):
    n_before = sum(1 for r in SeqIO.parse(core_fa, "fasta")) if os.path.exists(core_fa) else 4093
    n_after = sum(1 for r in SeqIO.parse(nr1_fa, "fasta"))
    print(f"Core proteins before CD-HIT: {n_before:,}")
    print(f"Non-redundant core proteins after CD-HIT: {n_after:,}")
else:
    print("[DEMO] Please ensure 02_Annotation/Core_OrthoGp_proteome_NR1.fa is present.")


## Step 6: Host Non-Homology Screening (*Bos taurus*)
Bacterial core proteins are compared against the **Bovine host proteome (`Database/BT`)** using BLASTP.
- **Cutoff:** Exclude any protein matching bovine host with **$	ext{Identity} > 40\%$ OR $	ext{Query Coverage} > 40\%$** ($E \le 10^{-20}$).

**Production command:**
```bash
blastp -query 02_Annotation/Core_OrthoGp_proteome_NR1.fa -db Database/BT -out 03_TargetHunt_Core/Core_BT.tsv -outfmt '6 qseqid pident qcovhsp evalue' -evalue 1e-20 -num_threads 25
```

In [ ]:
os.makedirs("03_TargetHunt_Core", exist_ok=True)
bt_tsv = "03_TargetHunt_Core/Core_BT.tsv"

if os.path.exists(bt_tsv):
    print(f"Loaded Bovine BLASTP results from: {bt_tsv}")
    # Parse tabular BLAST results
    df_bt = pd.read_csv(bt_tsv, sep="\t", header=None, names=["qseqid", "pident", "qcovhsp", "evalue"])
    
    # Filter proteins with Identity > 40% OR Coverage > 40%
    excluded_bt = set(df_bt.loc[(df_bt["pident"] > 40) | (df_bt["qcovhsp"] > 40), "qseqid"])
    with open("03_TargetHunt_Core/Core_BT_Exclude.ids", "w") as f:
        f.write("\n".join(sorted(excluded_bt)))
        
    print(f"Total bovine BLAST hits analyzed: {len(df_bt):,}")
    print(f"Homologous bacterial proteins excluded: {len(excluded_bt):,}")
    
    # Extract retained proteins
    retained_bt = [r for r in SeqIO.parse("02_Annotation/Core_OrthoGp_proteome_NR1.fa", "fasta") if r.id not in excluded_bt]
    n_bt = SeqIO.write(retained_bt, "03_TargetHunt_Core/Core_BT_NonHomo.fasta", "fasta")
    print(f"Bovine non-homologous proteins retained: {n_bt:,}")
else:
    print(f"[DEMO] {bt_tsv} not found. Please ensure 03_TargetHunt_Core/Core_BT.tsv is committed in your repo.")


## Step 7: Host Non-Homology Screening (*Homo sapiens*)
Screen the retained proteins against the **Human host proteome (`Database/HS`)** using BLASTP.
- **Cutoff:** Exclude any protein matching human host with **$	ext{Identity} > 40\%$ OR $	ext{Query Coverage} > 40\%$** ($E \le 10^{-20}$).

**Production command:**
```bash
blastp -query 03_TargetHunt_Core/Core_BT_NonHomo.fasta -db Database/HS -out 03_TargetHunt_Core/Core_HS.tsv -outfmt '6 qseqid pident qcovhsp evalue' -evalue 1e-20 -num_threads 25
```

In [ ]:
hs_tsv = "03_TargetHunt_Core/Core_HS.tsv"

if os.path.exists(hs_tsv):
    print(f"Loaded Human BLASTP results from: {hs_tsv}")
    df_hs = pd.read_csv(hs_tsv, sep="\t", header=None, names=["qseqid", "pident", "qcovhsp", "evalue"])
    
    excluded_hs = set(df_hs.loc[(df_hs["pident"] > 40) | (df_hs["qcovhsp"] > 40), "qseqid"])
    with open("03_TargetHunt_Core/Core_HS_Exclude.ids", "w") as f:
        f.write("\n".join(sorted(excluded_hs)))
        
    print(f"Total human BLAST hits analyzed: {len(df_hs):,}")
    print(f"Homologous bacterial proteins excluded: {len(excluded_hs):,}")
    
    # Extract final host non-homologs
    retained_hs = [r for r in SeqIO.parse("03_TargetHunt_Core/Core_BT_NonHomo.fasta", "fasta") if r.id not in excluded_hs]
    n_hs = SeqIO.write(retained_hs, "03_TargetHunt_Core/Core_HS_NonHomo.fasta", "fasta")
    print(f"Final host non-homologous proteins retained: {n_hs:,}")
else:
    print(f"[DEMO] {hs_tsv} not found. Please ensure 03_TargetHunt_Core/Core_HS.tsv is committed in your repo.")


## Step 8: Essentiality Screening Against DEG (Database of Essential Genes)
Identify essential bacterial proteins by comparing candidate proteins against the **Database of Essential Genes (`Database/DEGaa`)**.
- **Cutoff:** Retain proteins homologous to DEG entries with **$	ext{Identity} \ge 40\%$ AND $	ext{Query Coverage} \ge 40\%$** ($E \le 10^{-20}$).

**Production command:**
```bash
blastp -query 03_TargetHunt_Core/Core_HS_NonHomo.fasta -db Database/DEGaa -out 04_Druggability/Core_HS_DEG.tsv -outfmt '6 qseqid pident qcovhsp evalue' -evalue 1e-20 -num_threads 25
```

In [ ]:
os.makedirs("04_Druggability", exist_ok=True)
deg_tsv = "04_Druggability/Core_HS_DEG.tsv"

if os.path.exists(deg_tsv):
    print(f"Loaded DEG BLASTP results from: {deg_tsv}")
    df_deg = pd.read_csv(deg_tsv, sep="\t", header=None, names=["qseqid", "pident", "qcovhsp", "evalue"])
    
    # Retain matches meeting identity >= 40% AND coverage >= 40%
    essential_ids = set(df_deg.loc[(df_deg["pident"] >= 40) & (df_deg["qcovhsp"] >= 40), "qseqid"])
    with open("04_Druggability/Core_HS_DEG.ids", "w") as f:
        f.write("\n".join(sorted(essential_ids)))
        
    print(f"Candidate essential protein IDs: {len(essential_ids):,}")
    
    deg_records = [r for r in SeqIO.parse("03_TargetHunt_Core/Core_HS_NonHomo.fasta", "fasta") if r.id in essential_ids]
    n_deg = SeqIO.write(deg_records, "04_Druggability/Core_HS_DEG.fasta", "fasta")
    print(f"Essential proteins extracted: {n_deg:,}")
else:
    print(f"[DEMO] {deg_tsv} not found. Please ensure 04_Druggability/Core_HS_DEG.tsv is committed in your repo.")


## Step 9: Druggability Screening Against TTD (Therapeutic Target Database)
Identify druggable targets among essential proteins by searching against the **Therapeutic Target Database (`Database/TTD`)**.
- **Cutoff:** Retain proteins homologous to known drug targets with **$	ext{Identity} \ge 40\%$ AND $	ext{Query Coverage} \ge 40\%$** ($E \le 10^{-20}$).

**Production command:**
```bash
blastp -query 04_Druggability/Core_HS_DEG.fasta -db Database/TTD -out 04_Druggability/Core_HS_DEG_TTD.tsv -outfmt '6 qseqid pident qcovhsp evalue' -evalue 1e-20 -num_threads 25
```

In [ ]:
ttd_tsv = "04_Druggability/Core_HS_DEG_TTD.tsv"

if os.path.exists(ttd_tsv):
    print(f"Loaded TTD BLASTP results from: {ttd_tsv}")
    df_ttd = pd.read_csv(ttd_tsv, sep="\t", header=None, names=["qseqid", "pident", "qcovhsp", "evalue"])
    
    ttd_ids = set(df_ttd.loc[(df_ttd["pident"] >= 40) & (df_ttd["qcovhsp"] >= 40), "qseqid"])
    with open("04_Druggability/Core_HS_DEG_TTD.ids", "w") as f:
        f.write("\n".join(sorted(ttd_ids)))
        
    print(f"Candidate druggable target IDs: {len(ttd_ids):,}")
    
    ttd_records = [r for r in SeqIO.parse("04_Druggability/Core_HS_DEG.fasta", "fasta") if r.id in ttd_ids]
    n_ttd = SeqIO.write(ttd_records, "04_Druggability/Core_HS_DEG_TTD.fasta", "fasta")
    print(f"Druggable target proteins extracted: {n_ttd:,}")
else:
    print(f"[DEMO] {ttd_tsv} not found. Please ensure 04_Druggability/Core_HS_DEG_TTD.tsv is committed in your repo.")


## Step 10: Physicochemical Profiling & Stability Screening (ProtParam)
Calculate physicochemical properties (Molecular Weight, Isoelectric Point, Instability Index, GRAVY) for all druggable targets.
- **Cutoff:** Retain stable proteins with **$	ext{Instability Index} < 40$**.

In [ ]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis

input_fasta = "04_Druggability/Core_HS_DEG_TTD.fasta"
rows = []

if os.path.exists(input_fasta):
    for r in SeqIO.parse(input_fasta, "fasta"):
        try:
            cleaned = str(r.seq).replace("*", "")
            p = ProteinAnalysis(cleaned)
            rows.append([
                r.id,
                len(r.seq),
                round(p.molecular_weight(), 2),
                round(p.isoelectric_point(), 2),
                round(p.instability_index(), 2),
                round(p.gravy(), 3)
            ])
        except ValueError as e:
            print(f"Skipping {r.id}: {e}")

    df_prot = pd.DataFrame(rows, columns=[
        "Protein_ID", "Length", "Molecular_Weight",
        "Isoelectric_Point", "Instability_Index", "GRAVY"
    ])
    
    # Filter stable proteins (Instability Index < 40)
    stable_df = df_prot[df_prot["Instability_Index"] < 40]
    stable_df.to_csv("04_Druggability/Core_HS_DEG_TTD_ProtParam.csv", index=False)
    
    print(f"Total druggable targets analyzed: {len(df_prot):,}")
    print(f"Stable proteins (Instability Index < 40): {len(stable_df):,}")
    
    stable_ids = set(stable_df["Protein_ID"])
    stable_seqs = [r for r in SeqIO.parse(input_fasta, "fasta") if r.id in stable_ids]
    n_stable = SeqIO.write(stable_seqs, "04_Druggability/Core_HS_DEG_TTD_Stable.fasta", "fasta")
    print(f"Saved {n_stable} stable target sequences -> 04_Druggability/Core_HS_DEG_TTD_Stable.fasta")
else:
    print(f"[DEMO] {input_fasta} not found. Please ensure 04_Druggability/Core_HS_DEG_TTD.fasta is committed in your repo.")


## Step 11: Annotation Recovery & Gram Classification
1. Map candidate target IDs back to the original functional descriptions from the source proteomes.
2. Divide into:
   - **Gram-Positive:** *Staphylococcus*, *Streptococcus*
   - **Gram-Negative:** *Escherichia*, *Klebsiella*

In [ ]:
# 1. Collect descriptions from Proteome/*.faa
ann = {}
for f in glob.glob("Proteome/*.faa"):
    for r in SeqIO.parse(f, "fasta"):
        ann[r.id] = r.description

stable_fasta = "04_Druggability/Core_HS_DEG_TTD_Stable.fasta"
if os.path.exists(stable_fasta):
    rows = [(r.id, ann.get(r.id, "NOT_FOUND")) for r in SeqIO.parse(stable_fasta, "fasta")]
    df_ann = pd.DataFrame(rows, columns=["Protein_ID", "Annotation"])
    df_ann.to_csv("04_Druggability/Core_HS_DEG_TTD_Stable_Annotation.tsv", sep="\t", index=False)
    
    print(f"Successfully mapped annotations for {len(rows)} stable proteins.")
    
    # 2. Separate into Gram-Positive and Gram-Negative
    seqs = list(SeqIO.parse(stable_fasta, "fasta"))
    categories = [
        ("GramPositive", r"Staphylococcus|Streptococcus"),
        ("GramNegative", r"Escherichia|Klebsiella")
    ]
    
    for name, pattern in categories:
        ids = set(df_ann.loc[df_ann["Annotation"].str.contains(pattern, case=False, na=False), "Protein_ID"])
        matched = [r for r in seqs if r.id in ids]
        out_file = f"04_Druggability/Core_HS_DEG_TTD_Stable_{name}.fasta"
        n = SeqIO.write(matched, out_file, "fasta")
        print(f"- {name} targets extracted: {n} -> {out_file}")
else:
    print(f"[DEMO] {stable_fasta} not found.")


## Step 12: Prioritized Targets Summary & Subcellular Localization
Combine the physicochemical properties with functional annotations into a comprehensive summary table.

The generated FASTA files (`Core_HS_DEG_TTD_Stable_GramPositive.fasta` and `Core_HS_DEG_TTD_Stable_GramNegative.fasta`) can be submitted to the [PSORTb v3.0 Server](http://www.psort.org/psortb/) for subcellular localization predictions.

In [ ]:
if 'stable_df' in locals() and 'df_ann' in locals():
    summary_df = pd.merge(stable_df, df_ann, on="Protein_ID")
    summary_df.to_csv("04_Druggability/Final_Prioritized_Targets_Summary.csv", index=False)
    
    print("\n===== TOP 10 PRIORITIZED DRUG TARGETS =====")
    display(summary_df.head(10))
    
    # Check if PSORTb precomputed files exist
    psort_gp = glob.glob("*psortb*gp*txt") + glob.glob("psortb_gp_results*")
    psort_gn = glob.glob("*psortb*gn*txt") + glob.glob("psortb_gn_results*")
    if psort_gp or psort_gn:
        print("\n[INFO] PSORTb precomputed results found in repository!")
        print("Gram-Positive PSORTb file:", psort_gp)
        print("Gram-Negative PSORTb file:", psort_gn)
